# Modelo predictivo y visualización — Tipo C
## Análisis educativo de patrones sintéticos de sueño en recién nacidos

Este notebook genera datos ficticios, entrena un modelo de regresión y evalúa sus predicciones. No usa datos reales y no es una herramienta clínica.

## 1. Importar bibliotecas

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("Bibliotecas cargadas correctamente.")

## 2. Crear dataset sintético
Las relaciones se inventan exclusivamente para practicar. No representan recomendaciones ni valores esperados para bebés reales.

In [ ]:
rng = np.random.default_rng(42)
n = 800

edad_dias = rng.integers(0, 29, size=n)
tomas_24h = rng.integers(6, 15, size=n)
despertares_24h = rng.integers(3, 13, size=n)
siestas_24h = rng.integers(4, 11, size=n)

# Componentes inventados para generar un ejemplo reproducible
sueno_diurno_h = np.clip(
    rng.normal(7.0, 1.8, size=n) + 0.015 * edad_dias,
    1.0, 14.0
)
sueno_nocturno_h = np.clip(
    rng.normal(6.0, 1.7, size=n) - 0.01 * despertares_24h,
    1.0, 13.0
)
sueno_total_h = np.clip(
    sueno_diurno_h + sueno_nocturno_h + rng.normal(0, 0.25, size=n),
    2.0, 24.0
)

datos = pd.DataFrame({
    "edad_dias": edad_dias,
    "tomas_24h": tomas_24h,
    "despertares_24h": despertares_24h,
    "siestas_24h": siestas_24h,
    "sueno_diurno_h": np.round(sueno_diurno_h, 2),
    "sueno_nocturno_h": np.round(sueno_nocturno_h, 2),
    "sueno_total_h": np.round(sueno_total_h, 2),
})

display(datos.head())
print("Filas y columnas:", datos.shape)

## 3. Validar los datos

In [ ]:
print("Valores faltantes por columna:")
print(datos.isna().sum())

assert datos["edad_dias"].between(0, 28).all()
assert datos["sueno_diurno_h"].between(0, 24).all()
assert datos["sueno_nocturno_h"].between(0, 24).all()
assert datos["sueno_total_h"].between(0, 24).all()
assert (datos[["tomas_24h", "despertares_24h", "siestas_24h"]] >= 0).all().all()

print("\nValidaciones básicas completadas.")
print("Nota: los datos son sintéticos, no observaciones reales.")

## 4. Exploración inicial y visualización

In [ ]:
display(datos.describe().round(2))

plt.figure(figsize=(7, 4))
plt.hist(datos["sueno_total_h"], bins=20)
plt.title("Distribución del sueño total sintético")
plt.xlabel("Horas de sueño total (dato sintético)")
plt.ylabel("Frecuencia")
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4))
plt.scatter(datos["despertares_24h"], datos["sueno_total_h"], alpha=0.5)
plt.title("Sueño total sintético frente a despertares")
plt.xlabel("Despertares simulados en 24 horas")
plt.ylabel("Horas de sueño total sintéticas")
plt.tight_layout()
plt.show()

## 5. Preparar datos y entrenar modelo
Se predice `sueno_total_h` a partir de las demás variables. Se separa un 20% para prueba. Random Forest no requiere estandarización de las variables para este ejemplo.

In [ ]:
variables = [
    "edad_dias",
    "tomas_24h",
    "despertares_24h",
    "siestas_24h",
    "sueno_diurno_h",
    "sueno_nocturno_h",
]
X = datos[variables]
y = datos["sueno_total_h"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

modelo = RandomForestRegressor(
    n_estimators=150,
    max_depth=8,
    random_state=42
)
modelo.fit(X_train, y_train)

predicciones = modelo.predict(X_test)
print("Entrenamiento terminado.")
print("Filas de entrenamiento:", len(X_train))
print("Filas de prueba:", len(X_test))

## 6. Evaluación con métricas de regresión
- **MAE:** error absoluto medio, en horas.
- **RMSE:** raíz del error cuadrático medio, en horas.
- **R²:** compara el modelo con una predicción basada en la media del conjunto de prueba; puede ser negativo.

In [ ]:
mae = mean_absolute_error(y_test, predicciones)
rmse = np.sqrt(mean_squared_error(y_test, predicciones))
r2 = r2_score(y_test, predicciones)

print(f"MAE:  {mae:.3f} horas")
print(f"RMSE: {rmse:.3f} horas")
print(f"R²:   {r2:.3f}")

## 7. Gráfico: valores sintéticos frente a predicciones

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y_test, predicciones, alpha=0.6)

limite_min = min(y_test.min(), predicciones.min())
limite_max = max(y_test.max(), predicciones.max())
plt.plot([limite_min, limite_max], [limite_min, limite_max], linestyle="--")

plt.title("Valores reales sintéticos frente a predicciones")
plt.xlabel("Sueño total sintético observado (horas)")
plt.ylabel("Sueño total predicho (horas)")
plt.tight_layout()
plt.show()

## 8. Importancia de las variables

In [ ]:
importancias = pd.Series(
    modelo.feature_importances_,
    index=variables
).sort_values()

plt.figure(figsize=(8, 4))
importancias.plot(kind="barh")
plt.title("Importancia de variables según Random Forest")
plt.xlabel("Importancia relativa")
plt.tight_layout()
plt.show()

display(importancias.sort_values(ascending=False).to_frame("importancia"))

## 9. Ejemplo de predicción
El registro siguiente es ficticio. La predicción solo demuestra cómo llamar al modelo; no debe usarse para decidir cuánto debe dormir un bebé.

In [ ]:
nuevo_registro = pd.DataFrame([{
    "edad_dias": 12,
    "tomas_24h": 9,
    "despertares_24h": 7,
    "siestas_24h": 6,
    "sueno_diurno_h": 7.2,
    "sueno_nocturno_h": 6.1,
}])

estimacion = modelo.predict(nuevo_registro)[0]
print(f"Estimación del modelo sobre datos sintéticos: {estimacion:.2f} horas")
print("Advertencia: resultado educativo, no clínico.")

## 10. Conclusiones
1. Se construyó un dataset sintético y se comprobaron rangos básicos.
2. Se entrenó un modelo Random Forest de regresión.
3. Se evaluó con MAE, RMSE y R² y se generaron gráficos interpretativos.
4. Las métricas solo describen el rendimiento en este conjunto artificial. No permiten concluir nada sobre bebés reales ni diagnosticar trastornos del sueño.